In [2]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [ ]:

def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [4]:
import os
gemini = OpenAI(api_key=os.getenv('GEMINI_API_KEY'),base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [6]:
todos=[]
completed=[]

In [7]:
def get_todo_report() -> str:
    result=""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"

    show(result)
    return result

In [8]:
get_todo_report()

''

In [10]:
def create_todos(descriptions:list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False]*len(descriptions))
    return get_todo_report()

In [11]:
def mark_complete(index:int,completion_notes:str)->str:
    if 1<= index <= len(todos):
        completed[index-1]=True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [12]:
todos,completed=[],[]
create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [13]:
mark_complete(1,"bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [14]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}


In [15]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [16]:
tools=[{"type":"function","function":create_todos_json},
{"type":"function","function":mark_complete_json}]

In [18]:
def handle_tool_calls(tool_calls):
    results=[]
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role":"tool","content":json.dumps(result),"tool_call_id":tool_call.id})
    return results

In [19]:
from openai.types.responses import response_error


def loop(messages):
    done = False
    while not done:
        response = gemini.chat.completions.create(model="gemini-2.5-flash",messages=messages,tools=tools,reasoning_effort=None)
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [20]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [21]:
todos, completed = [], []
loop(messages)

Todo #1: Calculate the distance covered by the first train (from Boston) before the second train (from New York) 
starts.
Todo #2: Calculate the remaining distance between the two trains when the second train starts.
Todo #3: Calculate the combined speed of the two trains (relative speed).
Todo #4: Calculate the time it takes for them to meet after the second train starts.
Todo #5: Determine the meeting time by adding the calculated time to the second train's departure time.

The first train travels for 1 hour (from 2:00 PM to 3:00 PM) before the second train starts. Distance = 60 mph * 1 
hour = 60 miles.

Todo #1: Calculate the distance covered by the first train (from Boston) before the second train (from New York) 
starts.
Todo #2: Calculate the remaining distance between the two trains when the second train starts.
Todo #3: Calculate the combined speed of the two trains (relative speed).
Todo #4: Calculate the time it takes for them to meet after the second train starts.
Todo #5: Determine the meeting time by adding the calculated time to the second train's departure time.

Assuming a distance of 200 miles between Boston and New York, the remaining distance when the second train starts 
is 200 miles - 60 miles = 140 miles.

Todo #1: Calculate the distance covered by the first train (from Boston) before the second train (from New York) 
starts.
Todo #2: Calculate the remaining distance between the two trains when the second train starts.
Todo #3: Calculate the combined speed of the two trains (relative speed).
Todo #4: Calculate the time it takes for them to meet after the second train starts.
Todo #5: Determine the meeting time by adding the calculated time to the second train's departure time.

The combined speed of the two trains is 60 mph + 80 mph = 140 mph.

Todo #1: Calculate the distance covered by the first train (from Boston) before the second train (from New York) 
starts.
Todo #2: Calculate the remaining distance between the two trains when the second train starts.
Todo #3: Calculate the combined speed of the two trains (relative speed).
Todo #4: Calculate the time it takes for them to meet after the second train starts.
Todo #5: Determine the meeting time by adding the calculated time to the second train's departure time.

Time = Remaining Distance / Combined Speed = 140 miles / 140 mph = 1 hour.

Todo #1: Calculate the distance covered by the first train (from Boston) before the second train (from New York) 
starts.
Todo #2: Calculate the remaining distance between the two trains when the second train starts.
Todo #3: Calculate the combined speed of the two trains (relative speed).
Todo #4: Calculate the time it takes for them to meet after the second train starts.
Todo #5: Determine the meeting time by adding the calculated time to the second train's departure time.

The trains will meet at 4:00 PM.